# Download catalog files

In [ ]:
import os
if not os.path.exists("dja_msaexp_emission_lines_v4.4.csv"):
    ! wget https://zenodo.org/records/15472354/files/dja_msaexp_emission_lines_v4.4.csv.gz
    #decompress the file
    ! gunzip dja_msaexp_emission_lines_v4.4.csv.gz
if not os.path.exists("dja_msaexp_emission_lines_v4.4.columns.csv"):   
    ! wget https://zenodo.org/records/15472354/files/dja_msaexp_emission_lines_v4.4.columns.csv
    ! gunzip dja_msaexp_emission_lines_v4.4.columns.csv
    
if not os.path.exists("dja_msaexp_emission_lines_v4.5.csv"):   
    ! wget https://s3.amazonaws.com/msaexp-nirspec/extractions/dja_msaexp_emission_lines_v4.5.csv.gz
    ! gunzip dja_msaexp_emission_lines_v4.5.csv.gz
    
#https://s3.amazonaws.com/msaexp-nirspec/extractions/dja_msaexp_emission_lines_v4.5.prism_spectra.fits
if not os.path.exists("dja_msaexp_emission_lines_v4.5.prism_spectra.fits"):   
    ! wget https://s3.amazonaws.com/msaexp-nirspec/extractions/dja_msaexp_emission_lines_v4.5.prism_spectra.fits

In [ ]:
from astropy.table import Table
t = Table.read("dja_msaexp_emission_lines_v4.5.csv")
len(t[t['grating'] == 'PRISM'])

In [ ]:
from astropy.table import Table
t = Table.read("dja_msaexp_emission_lines_v4.5.prism_spectra.fits")
print(len(t['flux'][0]))
t

# Section 1 — Build merged spectra catalog `DJA_spectra_v4.5.fits`

The `*.prism_spectra.fits` file stores a **shared** wavelength grid (473 pts) and each
spectral quantity as a `(473, N_obj)` array. Column `i` ↔ the `i`-th **PRISM** row of the CSV
(verified: `wmin/wmax` agree to ~1e-7, `npix` agrees for all but 9 edge-pixel cases).

`build_dja_spectra_catalog.py` transposes the arrays to `(N_obj, 473)` and attaches them to the
PRISM metadata rows, producing a single file:

- HDU `CATALOG` — 42196 rows = all CSV metadata + vector cols `flux`/`err`/`full_err`/`valid_spec`, each `(473,)`
- HDU `WAVE` — the shared wavelength grid `(473,)`, micron

**No filtering** is applied — all PRISM rows are kept so filtering can happen downstream (e.g. in
the Dataset class). The spectral mask is stored as `valid_spec` to avoid clobbering the CSV's
scalar `valid` review-flag column.

In [ ]:
import os
# build once (skip if already present)
if not os.path.exists("DJA_spectra_v4.5.fits"):
    !python build_dja_spectra_catalog.py

In [ ]:
# --- load: a few lines pull out any spectrum, aligned with all metadata ---
from astropy.table import Table
from astropy.io import fits
import matplotlib.pyplot as plt

cat  = Table.read("DJA_spectra_v4.5.fits", hdu="CATALOG")   # 42196 rows, metadata + spectra
wave = fits.getdata("DJA_spectra_v4.5.fits", "WAVE")        # shared grid (473,), micron
print(len(cat), "spectra |", len(cat.colnames), "columns")
print("spectral cols: flux/err/full_err/valid_spec, each", cat["flux"].shape)

i = 1050
v = cat["valid_spec"][i]
plt.figure(figsize=(8,3))
plt.plot(wave[v], cat["flux"][i][v], drawstyle="steps-mid", lw=0.8)
plt.fill_between(wave[v], (cat["flux"][i]-cat["full_err"][i])[v], (cat["flux"][i]+cat["full_err"][i])[v],
                 alpha=0.25, step="mid")
plt.xlabel("wave (micron)"); plt.ylabel("flux (uJy)")
plt.title(f"{cat['file'][i]}  z={cat['z_best'][i]:.3f}  grade={cat['grade'][i]}  sn50={cat['sn50'][i]:.2f}")
plt.tight_layout(); plt.show()

# Section 2 — Per-resolution download → `.h5` export pipeline

End-to-end flow that turns the emission-line catalog into ML-ready per-resolution HDF5 files:

1. **Setup** — load the catalog as a pandas `df`, set the download dir.
2. **Filter masks** — quality cut (`grade`, `obs_365_frac`) split by grating into low / mid / high resolution.
3. **Save** one CSV per resolution.
4. **Download** the individual `.spec.fits` files referenced by each CSV.
5. **Export** each resolution to a single `dja_spectra_{res}.h5`.
6. **Visualize** random example spectra per resolution.

In [ ]:
# 2.0  Setup — load catalog as pandas df + common config
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy.table import Table

download_dir = "download"            # individual .spec.fits land here
df = pd.read_csv("dja_msaexp_emission_lines_v4.5.csv", low_memory=False)
print(f"{len(df)} rows, {len(df.columns)} columns")

In [ ]:
# 2.1  Filter masks (quality cut + split by grating resolution) + S/N histogram
import matplotlib.pyplot as plt
import numpy as np

#unique filters and gratings
print("Unique filters:", df['filter'].unique())
print("Unique gratings:", df['grating'].unique())

mask_main = df['grade'].isin([1,2,3]) & (df['obs_365_frac'] > 0.5) #& (df["wmin"] < 1.3) & (df["wmax"] > 1.7)
mask_low_res = mask_main  & df['grating'].isin(['PRISM'])
mask_mid_res = mask_main  & (df['grating'].isin(['G140M', 'G235M', 'G395M']))
mask_high_res = mask_main & (df['grating'].isin(['G140H', 'G235H', 'G395H']))

sn_median = df["sn50"][mask_main].dropna()

plt.figure(figsize=(6,4))
plt.hist(sn_median, bins=50, histtype="step", color="C0", range=(0, 12), cumulative=True, density=True)
plt.xlabel("Median S/N (sn50)")
plt.ylabel("Count")
plt.title("Histogram of median S/N")
plt.tight_layout()
plt.show()

print("length of low res spectra:", mask_low_res.sum())
print('length of low res spectra with grade >= 2', (mask_low_res & (df['grade'] >= 2)).sum())
print("length of mid res spectra:", mask_mid_res.sum())
print('length of mid res spectra with grade >= 2', (mask_mid_res & (df['grade'] >= 2)).sum())
print("length of high res spectra:", mask_high_res.sum())
print('length of high res spectra with grade >= 2', (mask_high_res & (df['grade'] >= 2)).sum())

In [ ]:
# 2.2  Save one CSV per resolution
#save csv per resolution
if not os.path.exists("dja_low_res.csv"):
    df[mask_low_res].to_csv("dja_low_res.csv", index=False)
if not os.path.exists("dja_mid_res.csv"):
    df[mask_mid_res].to_csv("dja_mid_res.csv", index=False)
if not os.path.exists("dja_high_res.csv"):
    df[mask_high_res].to_csv("dja_high_res.csv", index=False)
    
manual_refresh = True
if manual_refresh:
    df[mask_low_res].to_csv("dja_low_res.csv", index=False)
    df[mask_mid_res].to_csv("dja_mid_res.csv", index=False)
    df[mask_high_res].to_csv("dja_high_res.csv", index=False)

In [ ]:
# 2.3  Download the individual .spec.fits files
#run download spectra
!python download_spectra_from_csv.py --csv-table 'dja_low_res.csv'  --num-workers 20
!python download_spectra_from_csv.py --csv-table 'dja_mid_res.csv'  --num-workers 20
!python download_spectra_from_csv.py --csv-table 'dja_high_res.csv' --num-workers 20

In [ ]:
# 2.4  Export each resolution to a single dja_spectra_{res}.h5
#make separate .h5 files
#!python export_dja_spectrum.py --csv 'dja_low_res.csv'  --num-workers 20 --out 'dja_spectra_low.h5'
#!python export_dja_spectrum.py --csv 'dja_mid_res.csv'  --num-workers 20 --out 'dja_spectra_mid.h5'
#!python export_dja_spectrum.py --csv 'dja_high_res.csv' --num-workers 20 --out 'dja_spectra_high.h5'

In [ ]:
# 2.5  Visualize random example spectra per resolution
#visualize n examples from each .h5 file
import h5py

n_sample = 36
for res in ["low", "mid", "high"]:
    with h5py.File(f"dja_spectra_{res}.h5", "r") as f:
        print(f"Resolution: {res}, shape of flux: {f['flux'].shape}, number of spectra: {len(f['flux'])}, wavelength range: {f['wave'][0][0]} - {f['wave'][0][-1]}")
        #filter out exazmple spectra with sn50 < 5
        mask_sn = f["sn50"][:] >= 1
        #random n sample index
        idx = np.random.choice(np.where(mask_sn)[0], n_sample)
        
        #plot sn cumulative histogram
        plt.figure(figsize=(6,4))
        plt.hist(f["sn50"][:], bins=50, histtype="step", color="C0", range=(0, 12), cumulative=True, density=False)
        plt.xlabel("Median S/N (sn50)")
        plt.ylabel("Count")
        plt.title(f"Histogram of median S/N for {res} res spectra")
        plt.tight_layout()
        plt.show()
        
        #4 columns of subplots
        rows = (n_sample + 3) // 4
        fig, axes = plt.subplots(rows, 4, figsize=(24, 4*rows))
        for i, index in enumerate(idx):
            ax = axes.flatten()[i]
            wave = f["wave"][index]
            flux = f["flux"][index]
            ax.plot(wave, flux)
            ax.set_title(f"sn50: {f['sn50'][index]:.2f}")
        plt.tight_layout()
        plt.show()
            

In [ ]:
# 2.6  Same, restricted to grade-3 spectra
#plot same example plots with grade 3
for res in ["low", "mid", "high"]:
    with h5py.File(f"dja_spectra_{res}.h5", "r") as f:
        print(f"Resolution: {res}, shape of flux: {f['flux'].shape}, number of spectra: {len(f['flux'])}, wavelength range: {f['wave'][0][0]} - {f['wave'][0][-1]}")
        #filter out exazmple spectra with sn50 < 5 and grade == 3
        mask_sn = f["sn50"][:] >= 3
        mask_grade = f["grade"][:] == 3
        mask = mask_sn & mask_grade
        #random n sample index
        idx = np.random.choice(np.where(mask)[0], n_sample)
        
        #plot sn cumulative histogram
        plt.figure(figsize=(6,4))
        plt.hist(f["sn50"][:], bins=50, histtype="step", color="C0", range=(0, 12), cumulative=True, density=False)
        plt.xlabel("Median S/N (sn50)")
        plt.ylabel("Count")
        plt.title(f"Histogram of median S/N for grade 3 {res} res spectra")
        plt.tight_layout()
        plt.show()
        
        #4 columns of subplots
        rows = (n_sample + 3) // 4
        fig, axes = plt.subplots(rows, 4, figsize=(24, 4*rows))
        for i, index in enumerate(idx):
            ax = axes.flatten()[i]
            wave = f["wave"][index]
            flux = f["flux"][index]
            ax.plot(wave, flux)
            ax.set_title(f"sn50: {f['sn50'][index]:.2f}")
        plt.tight_layout()
        plt.show()